## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 61 samples and 13399 columns.
First few rows of the data:


,os time (months),OS status,sample_title,tissue,age(0,genotype(1,tumor size (0,t stage (1,n stage (0,m stage (0,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,93.700000,0.0,TFE3-1,Renal cell carcinoma,<18; 1: ≥18): 0,male; 2: female): 1,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 0,m0; 1:m1): 0,...,5.308855,3.757621,5.360897,5.656186,3.962839,4.470927,6.469223,6.748252,6.125305,5.764987
1,104.800000,0.0,TFE3-2,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 1,t1; 2: t2; 3: t3; 4:t4): 3,n0; 1:n1): 0,m0; 1:m1): 0,...,5.848443,3.730599,5.919254,6.527469,4.660299,4.747210,6.781149,6.862246,4.832972,6.616324
2,52.333333,1.0,TFE3-3,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 1,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 0,m0; 1:m1): 1,...,7.678366,5.906810,7.449775,7.975221,5.812916,6.476641,8.494724,7.960829,6.346937,7.731272
3,71.900000,0.0,TFE3-4,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 1,m0; 1:m1): 0,...,6.451027,4.863556,6.369385,6.755879,5.045029,6.036748,7.921962,8.600118,7.713849,7.892062
4,10.000000,1.0,TFE3-5,Renal cell carcinoma,<18; 1: ≥18): 0,male; 2: female): 1,<median; 1: ≥median): 1,t1; 2: t2; 3: t3; 4:t4): 4,n0; 1:n1): 0,m0; 1:m1): 0,...,6.448179,4.869214,6.499521,7.077482,4.883677,5.596320,7.486865,7.328116,6.006584,6.856885


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
